In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


# Phase 7 — Cluster Profiling & Business Analysis

**Project:** Wholesale Customer Segmentation

Phase 7 - Cluster Profiling & Business Analysis
Wholesale Customers Clustering Analysis

Steps covered (per implementation plan):
 18.   Measure cluster sizes
 19.   Build cluster profiles
 19.5. Rank features by cluster-discriminating power
 20.   Compare clusters by spending behavior
 21.   Analyze Channel and Region after clustering

Depends on: phase1_setup.py, phase2_eda.py (SPEND_COLS, FIG_DIR),
            phase6_train_validate.py output (labeled_customers.csv)

Assumption (explicitly stated, carried from Phase 6's evidence):
 K=2 is the primary solution because it has the highest silhouette score and is stable.
 K=3 is retained only as a secondary comparison because its silhouette is lower, although
 it also clears the project's stability threshold. K=4 is more weakly supported because its
 bootstrap stability falls below the threshold.

### How to use this notebook
Run cells from top to bottom. Keep the project files in the same folder as this notebook. Phases 1–4 create the data preparation artifacts used by later phases; Phases 5–9 read those artifacts; Phase 10 assembles the final report.

In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway



# Recreate constants locally. Phase 6 supplies labeled_customers.csv.
SPEND_COLS = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
FIG_DIR = "figures"
import os
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_style("whitegrid")

PRIMARY_CLUSTER_COL = "cluster_k2"
SECONDARY_CLUSTER_COL = "cluster_k3"

## SECTION 1: Load Labeled Data

In [ ]:
# SECTION 1: Load Labeled Data
# ===========================================================================
def load_labeled_data() -> pd.DataFrame:
    """Load the raw data with cluster labels attached (from Phase 6)."""
    print("=" * 70)
    print("SECTION 1: LOAD LABELED DATA")
    print("=" * 70)

    df = pd.read_csv("labeled_customers.csv")
    print(f"Loaded labeled_customers.csv — shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"Primary clustering for this phase: '{PRIMARY_CLUSTER_COL}' "
          "(stable, per Phase 6).")
    print(f"Secondary/exploratory clustering: '{SECONDARY_CLUSTER_COL}' "
          "(stable but lower-silhouette alternative — shown briefly).")
    return df

## SECTION 2: Cluster Sizes (Step 18)

In [ ]:
# SECTION 2: Cluster Sizes (Step 18)
# ===========================================================================
def report_cluster_sizes(df: pd.DataFrame, cluster_col: str) -> pd.DataFrame:
    """Report cluster sizes as counts and percentage of the total 440 customers."""
    print("\n" + "=" * 70)
    print(f"SECTION 2: CLUSTER SIZES ({cluster_col})")
    print("=" * 70)

    sizes = df[cluster_col].value_counts().sort_index()
    pct = (sizes / len(df) * 100).round(1)
    size_table = pd.DataFrame({"count": sizes, "pct_of_440": pct})
    print(size_table.to_string())
    return size_table

## SECTION 3: Build Per-Cluster Profiles (Step 19)

In [ ]:
# SECTION 3: Build Per-Cluster Profiles (Step 19)
# ===========================================================================
def build_cluster_profiles(df: pd.DataFrame, cluster_col: str) -> pd.DataFrame:
    """Compute mean and median spend per category, per cluster."""
    print("\n" + "=" * 70)
    print(f"SECTION 3: PER-CLUSTER PROFILES ({cluster_col})")
    print("=" * 70)

    mean_profile = df.groupby(cluster_col)[SPEND_COLS].mean().round(1)
    median_profile = df.groupby(cluster_col)[SPEND_COLS].median().round(1)

    print("Mean spend per category, per cluster:")
    print(mean_profile.to_string())
    print("\nMedian spend per category, per cluster:")
    print(median_profile.to_string())

    return mean_profile


def plot_cluster_profile_bars(mean_profile: pd.DataFrame, cluster_col: str) -> None:
    """Grouped bar chart of mean spend per category, per cluster."""
    profile_pct = mean_profile.T  # categories as rows, clusters as columns
    fig, ax = plt.subplots(figsize=(11, 6))
    profile_pct.plot(kind="bar", ax=ax, colormap="Set2")
    ax.set_title(f"Mean Spend per Category by Cluster ({cluster_col})")
    ax.set_ylabel("Mean Annual Spend")
    ax.set_xlabel("Product Category")
    ax.legend(title="Cluster")
    plt.xticks(rotation=30, ha="right")
    fig.tight_layout()
    fname = f"{FIG_DIR}/14_cluster_profile_bars_{cluster_col}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {fname}")


def plot_cluster_profile_radar(mean_profile: pd.DataFrame, cluster_col: str) -> None:
    """Radar chart of mean spend per category, per cluster, using
    min-max normalized values so all categories share a comparable 0-1 scale
    (raw spend scales differ by an order of magnitude across categories).

    CAVEAT (K=2 specific): with exactly 2 clusters, min-max normalization
    mathematically forces one cluster to exactly 1.0 and the other to
    exactly 0.0 on every single category, regardless of how large or small
    the real gap is. This produces a visually dramatic 'hourglass' shape
    that shows DIRECTION (which cluster is higher) but destroys MAGNITUDE
    information. The grouped bar chart (Section 3) is the reliable source
    for actual spend levels and gap sizes; this radar chart is included
    because the plan calls for a radar-or-bar chart, but should be read
    only as a direction summary, not a magnitude comparison, when K=2."""
    normalized = (mean_profile - mean_profile.min()) / (mean_profile.max() - mean_profile.min())
    categories = list(normalized.columns)
    n_cats = len(categories)
    angles = np.linspace(0, 2 * np.pi, n_cats, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    colors = sns.color_palette("Set2", len(normalized))
    for i, (cluster_id, row) in enumerate(normalized.iterrows()):
        values = row.tolist()
        values += values[:1]
        ax.plot(angles, values, label=f"Cluster {cluster_id}", color=colors[i])
        ax.fill(angles, values, alpha=0.15, color=colors[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    ax.set_title(f"Cluster Spend Profiles (Normalized) — {cluster_col}")
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    fig.tight_layout()
    fname = f"{FIG_DIR}/15_cluster_profile_radar_{cluster_col}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {fname}")
    print("[NOTE] Radar values are min-max normalized per category (0=lowest "
          "cluster mean, 1=highest) purely for visual comparability — refer "
          "to Section 3's raw mean/median table for actual spend figures.")
    if len(normalized) == 2:
        print("[CAVEAT] With exactly 2 clusters, min-max normalization forces "
              "every category to exactly 0.0/1.0 regardless of the true gap "
              "size — this radar chart shows DIRECTION only, not magnitude. "
              "Use the grouped bar chart (Section 3) for actual spend "
              "comparisons; do not read the radar's shape as proportional "
              "to how different the clusters really are.")

## SECTION 4: Rank Features by Cluster-Discriminating Power (Step 19.5)

In [ ]:
# SECTION 4: Rank Features by Cluster-Discriminating Power (Step 19.5)
# ===========================================================================
def rank_discriminating_features(df: pd.DataFrame, cluster_col: str) -> pd.DataFrame:
    """Rank features by descriptive separation strength using ANOVA F and eta-squared.

    The clusters were created from these same variables, so p-values are not treated
    as independent confirmatory evidence.
    """
    print("\n" + "=" * 70)
    print(f"SECTION 4: DESCRIPTIVE FEATURE SEPARATION ({cluster_col})")
    print("=" * 70)
    rows = []
    cluster_ids = sorted(df[cluster_col].unique())
    for col in SPEND_COLS:
        groups = [df.loc[df[cluster_col] == cid, col].values for cid in cluster_ids]
        f_stat, p_val = f_oneway(*groups)
        grand_mean = df[col].mean()
        ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
        ss_total = ((df[col] - grand_mean) ** 2).sum()
        eta2 = ss_between / ss_total if ss_total else 0.0
        rows.append({"feature": col, "F_statistic": f_stat, "eta_squared": eta2, "p_value": p_val})
    ranking = pd.DataFrame(rows).sort_values("F_statistic", ascending=False).reset_index(drop=True)
    ranking.index = ranking.index + 1
    print(ranking.round(4).to_string())
    print("\n[NOTE] F-statistic and eta-squared are descriptive because the clusters were constructed from these variables.")
    return ranking


def plot_feature_ranking(ranking: pd.DataFrame, cluster_col: str) -> None:
    """Bar chart of F-statistics per feature, ordered by discriminating power."""
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(ranking["feature"][::-1], ranking["F_statistic"][::-1], color="teal")
    ax.set_xlabel("Descriptive ANOVA F-statistic")
    ax.set_title(f"Observed Feature Separation by Cluster ({cluster_col})")
    fig.tight_layout()
    fname = f"{FIG_DIR}/16_feature_discriminating_power_{cluster_col}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {fname}")

## SECTION 5: Compare High-Spend vs. Low-Spend Clusters (Step 20)

In [ ]:
# SECTION 5: Compare High-Spend vs. Low-Spend Clusters (Step 20)
# ===========================================================================
def compare_high_low_spend_clusters(mean_profile: pd.DataFrame, cluster_col: str) -> dict:
    """Identify which cluster is the 'high total spend' cluster vs. 'low
    total spend' cluster, and which categories drive the separation between
    them."""
    print("\n" + "=" * 70)
    print(f"SECTION 5: HIGH-SPEND VS. LOW-SPEND CLUSTER COMPARISON ({cluster_col})")
    print("=" * 70)

    total_spend = mean_profile.sum(axis=1).sort_values(ascending=False)
    print(f"Total mean spend per cluster (sum across all 6 categories):")
    print(total_spend.round(1).to_string())

    high_cluster = total_spend.index[0]
    low_cluster = total_spend.index[-1]
    print(f"\nHighest total-spend cluster: {high_cluster} "
          f"(total mean spend = {total_spend.iloc[0]:.1f})")
    print(f"Lowest total-spend cluster: {low_cluster} "
          f"(total mean spend = {total_spend.iloc[-1]:.1f})")

    diff = (mean_profile.loc[high_cluster] - mean_profile.loc[low_cluster]).sort_values(ascending=False)
    print(f"\nCategory-level difference (high-spend cluster minus low-spend cluster), "
          "largest gap first:")
    print(diff.round(1).to_string())
    print(f"\n[FINDING] The category driving the largest separation between the "
          f"high- and low-spend clusters is '{diff.index[0]}' "
          f"(gap = {diff.iloc[0]:.1f}).")

    return {"high_cluster": high_cluster, "low_cluster": low_cluster, "category_gaps": diff}

## SECTION 6: Cross-Tabulate Clusters Against Channel & Region (Step 21)

In [ ]:
# SECTION 6: Cross-Tabulate Clusters Against Channel & Region (Step 21)
# ===========================================================================
def crosstab_channel_region(df: pd.DataFrame, cluster_col: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Cross-tabulate cluster membership against Channel and Region (both
    held out of the clustering feature set in Phase 2) as a sanity check on
    whether clusters align with the known Horeca/Retail channel split."""
    print("\n" + "=" * 70)
    print(f"SECTION 6: CROSS-TABULATE CLUSTERS VS. CHANNEL & REGION ({cluster_col})")
    print("=" * 70)

    channel_map = {1: "Horeca (1)", 2: "Retail (2)"}
    channel_ct = pd.crosstab(df[cluster_col], df["Channel"].map(channel_map))
    channel_ct_pct = (channel_ct.div(channel_ct.sum(axis=1), axis=0) * 100).round(1)
    print("Cluster x Channel (counts):")
    print(channel_ct.to_string())
    print("\nCluster x Channel (row %):")
    print(channel_ct_pct.to_string())

    region_ct = pd.crosstab(df[cluster_col], df["Region"])
    region_ct_pct = (region_ct.div(region_ct.sum(axis=1), axis=0) * 100).round(1)
    print("\nCluster x Region (counts):")
    print(region_ct.to_string())
    print("\nCluster x Region (row %):")
    print(region_ct_pct.to_string())

    # Quantify channel alignment: for each cluster, what % belongs to its
    # majority channel? High values (>>50%) suggest clusters mirror Channel.
    majority_pct = channel_ct_pct.max(axis=1)
    print(f"\n[FINDING] Majority-channel share per cluster: {majority_pct.round(1).to_dict()}")
    if (majority_pct > 70).all():
        print("[FINDING] Clusters show strong alignment with the existing Channel "
              "variable (each cluster is dominated by one channel) — clustering "
              "on spend alone largely rediscovers the known Horeca/Retail split.")
    else:
        print("[FINDING] Clusters do NOT strongly mirror Channel — spend-based "
              "segmentation reveals structure beyond the existing channel label.")

    return channel_ct, region_ct


def plot_channel_crosstab(channel_ct: pd.DataFrame, cluster_col: str) -> None:
    """Stacked bar chart of Channel composition per cluster."""
    fig, ax = plt.subplots(figsize=(7, 5))
    channel_ct.plot(kind="bar", stacked=True, ax=ax, colormap="Set2")
    ax.set_title(f"Channel Composition per Cluster ({cluster_col})")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Number of Customers")
    ax.legend(title="Channel")
    plt.xticks(rotation=0)
    fig.tight_layout()
    fname = f"{FIG_DIR}/17_channel_crosstab_{cluster_col}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {fname}")


def plot_region_crosstab(region_ct: pd.DataFrame, cluster_col: str) -> None:
    """Stacked bar chart of Region composition per cluster."""
    fig, ax = plt.subplots(figsize=(8, 5))
    region_ct.plot(kind="bar", stacked=True, ax=ax, colormap="Pastel1")
    ax.set_title(f"Region Composition per Cluster ({cluster_col})")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Number of Customers")
    ax.legend(title="Region")
    plt.xticks(rotation=0)
    fig.tight_layout()
    fname = f"{FIG_DIR}/18_region_crosstab_{cluster_col}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {fname}")

## SECTION 7: Brief Secondary Comparison — K=3 (Exploratory)

In [ ]:
# SECTION 7: Brief Secondary Comparison — K=3 (Exploratory)
# ===========================================================================
def brief_secondary_k3_summary(df: pd.DataFrame) -> None:
    """Brief profile of K=3 for reference only. K=3 clears the project's stability threshold
    but has lower silhouette than K=2, so it is treated as a secondary, more-granular alternative."""
    print("\n" + "=" * 70)
    print("SECTION 7: BRIEF SECONDARY COMPARISON — K=3 (EXPLORATORY)")
    print("=" * 70)
    print("[NOTE] K=3 is stable under the project's threshold but has lower silhouette than K=2. "
          "It is shown as a secondary, more-granular alternative rather than the primary solution.")

    sizes = df[SECONDARY_CLUSTER_COL].value_counts().sort_index()
    pct = (sizes / len(df) * 100).round(1)
    print(f"\nK=3 cluster sizes: {sizes.to_dict()} (% of 440: {pct.to_dict()})")

    mean_profile_k3 = df.groupby(SECONDARY_CLUSTER_COL)[SPEND_COLS].mean().round(1)
    print("\nK=3 mean spend per category, per cluster:")
    print(mean_profile_k3.to_string())

## MAIN — run Phase 7 end to end

In [ ]:
# MAIN — run Phase 7 end to end
# ===========================================================================
if __name__ == "__main__":
    df = load_labeled_data()

    # --- Primary analysis: K=2 (stable, per Phase 6) ---
    size_table = report_cluster_sizes(df, PRIMARY_CLUSTER_COL)
    mean_profile = build_cluster_profiles(df, PRIMARY_CLUSTER_COL)
    plot_cluster_profile_bars(mean_profile, PRIMARY_CLUSTER_COL)
    print("[NOTE] Radar chart omitted from the final report because K=2 min-max normalization forces each category to 0/1 and exaggerates magnitude differences.")

    ranking = rank_discriminating_features(df, PRIMARY_CLUSTER_COL)
    plot_feature_ranking(ranking, PRIMARY_CLUSTER_COL)

    high_low = compare_high_low_spend_clusters(mean_profile, PRIMARY_CLUSTER_COL)

    channel_ct, region_ct = crosstab_channel_region(df, PRIMARY_CLUSTER_COL)
    plot_channel_crosstab(channel_ct, PRIMARY_CLUSTER_COL)
    plot_region_crosstab(region_ct, PRIMARY_CLUSTER_COL)

    # --- Secondary, brief comparison: K=3 ---
    brief_secondary_k3_summary(df)

    print("\n" + "=" * 70)
    print("PHASE 7 COMPLETE")
    print("=" * 70)
    print(f"[OK] Cluster sizes reported for {PRIMARY_CLUSTER_COL}.")
    print("[OK] Per-cluster profiles built using mean/median spend and descriptive separation analysis.")
    print(f"[OK] Features ranked by discriminating power: top feature = "
          f"'{ranking.iloc[0]['feature']}'.")
    print(f"[OK] High-spend cluster = {high_low['high_cluster']}, "
          f"low-spend cluster = {high_low['low_cluster']}; top separating "
          f"category = '{high_low['category_gaps'].index[0]}'.")
    print("[OK] Channel/Region cross-tabulation complete.")
    print("[OK] K=3 shown briefly as stable but lower-silhouette secondary reference.")
    print("[OK] Ready for Phase 8 (PCA Dimensionality Reduction & Visualization).")

### Phase 7 checkpoint

Review the outputs and figures generated by this phase before moving to the next phase.